# Peptide Discovery v2 — Google Colab (T4 GPU)

**実行手順:**
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選択
2. Cell 1〜6 を順に実行（Cell 6 は完了したら Cell 7 へ）
3. **Cell 7 を実行したままにする**（アイドル切断防止 + URL 表示）

> Boltz モデルウェイト (~5 GB) を Google Drive にキャッシュすることで、2回目以降の起動を高速化できます。

In [ ]:
# ── Cell 1: GPU 確認 ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU 検出:', result.stdout.strip())
else:
    print('❌ GPU が検出されません。ランタイムのタイプを T4 GPU に変更してください。')
    raise SystemExit('GPU required')

In [ ]:
# ── Cell 2: Google Drive マウント（モデルウェイトのキャッシュ用）────────────────
USE_DRIVE_CACHE = True  # False にするとセッションごとに再ダウンロード

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    BOLTZ_CACHE_DIR = '/content/drive/MyDrive/boltz_cache'
    import os
    os.makedirs(BOLTZ_CACHE_DIR, exist_ok=True)
    if not os.path.exists(os.path.expanduser('~/.boltz')):
        os.symlink(BOLTZ_CACHE_DIR, os.path.expanduser('~/.boltz'))
    print('✅ Drive キャッシュ設定完了:', BOLTZ_CACHE_DIR)
else:
    print('ℹ️  Drive キャッシュなし（セッションごとに再ダウンロード）')

In [ ]:
# ── Cell 3: リポジトリ取得 ────────────────────────────────────────────────────
import getpass, os

GITHUB_USER  = 'rkikuchi-bis'
GITHUB_REPO  = 'peptide_v2'
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token: ')
repo_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

if os.path.exists(f'/content/{GITHUB_REPO}'):
    print('既存のディレクトリを更新します...')
    !cd /content/{GITHUB_REPO} && git pull
else:
    !git clone {repo_url} /content/{GITHUB_REPO}

os.chdir(f'/content/{GITHUB_REPO}')
print('✅ 作業ディレクトリ:', os.getcwd())

In [ ]:
# ── Cell 4: 依存関係インストール ──────────────────────────────────────────────
import subprocess, sys

packages = [
    'biopython==1.84',
    'streamlit>=1.36.0',
    'py3dmol>=2.0.0',
    'gemmi>=0.6',
    'altair>=5.0',
    'huggingface-hub>=0.20',
    'einops>=0.7',
    'biotite>=0.38',
    'boltz==2.2.1',
    'prodigy-prot',
]

print('パッケージをインストール中...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('インストール失敗')
print('✅ インストール完了')

In [ ]:
# ── Cell 5: Boltz モデルウェイト事前ダウンロード ──────────────────────────────
import os, subprocess, tempfile, yaml
from pathlib import Path

ckpt_path = Path(os.path.expanduser('~/.boltz')) / 'boltz1_conf.ckpt'

if ckpt_path.exists():
    print(f'✅ キャッシュ済み ({ckpt_path.stat().st_size/1e9:.1f} GB) — スキップ')
else:
    print('モデルウェイトをダウンロード中（初回のみ、約 5〜10 分）...')
    dummy_yaml = {'version': 1, 'sequences': [{'protein': {'id': 'A', 'sequence': 'ACDEFGHIKLMNPQRSTVWY'}}]}
    with tempfile.TemporaryDirectory() as tmpdir:
        yaml_path = Path(tmpdir) / 'dummy.yaml'
        yaml_path.write_text(yaml.dump(dummy_yaml))
        subprocess.run(['boltz', 'predict', str(yaml_path), '--out_dir', tmpdir,
                        '--model', 'boltz1', '--accelerator', 'gpu',
                        '--sampling_steps', '1', '--use_msa_server'],
                       capture_output=True)
    print('✅ ダウンロード完了' if ckpt_path.exists() else '⚠️  失敗。Cell 6 で自動DLされます。')

In [ ]:
# ── Cell 6: Streamlit + cloudflared 起動（すぐに完了します）──────────────────
# このセルは起動後すぐに完了します。プロセスはバックグラウンドで動き続けます。

import subprocess, time, re, sys, os
from pathlib import Path

PORT = 8501
CLOUDFLARED = Path('/usr/local/bin/cloudflared')

# cloudflared バイナリ取得
if not CLOUDFLARED.exists():
    print('cloudflared をダウンロード中...')
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(CLOUDFLARED)], check=True)
    CLOUDFLARED.chmod(0o755)

# 既存プロセスを停止してから再起動
os.system('pkill -f "streamlit run" 2>/dev/null; pkill -f cloudflared 2>/dev/null')
time.sleep(2)

# Streamlit をバックグラウンド起動
with open('/tmp/streamlit.log', 'w') as log:
    st_proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py',
         '--server.port', str(PORT),
         '--server.headless', 'true',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=log, stderr=log
    )
Path('/tmp/st_pid').write_text(str(st_proc.pid))
time.sleep(5)

# cloudflared トンネル起動
# stderr をファイルへ書き出す（subprocess.PIPE はバッファが数分で溢れてクラッシュするため使わない）
Path('/tmp/cloudflared.log').write_text('')
with open('/tmp/cloudflared.log', 'w') as cf_log:
    cf_proc = subprocess.Popen(
        [str(CLOUDFLARED), 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=subprocess.DEVNULL,
        stderr=cf_log
    )
Path('/tmp/cf_pid').write_text(str(cf_proc.pid))

# URL 取得: ファイルをポーリング（最大 30 秒）
url = None
for _ in range(30):
    time.sleep(1)
    try:
        text = Path('/tmp/cloudflared.log').read_text()
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', text)
        if m:
            url = m.group(0)
            break
    except Exception:
        pass

Path('/tmp/app_url').write_text(url or '')

print('='*60)
print('✅ 起動完了！')
print(f'   URL: {url or "取得失敗 → Cell 6 を再実行"}')
print('='*60)
print('次に Cell 7 を実行してください（アイドル切断防止）。')


In [ ]:
# ── Cell 7: アイドル切断防止 + 監視（実行したままにしてください）──────────────
# Cell 6 のプロセスはバックグラウンドで稼働中です。
# このセルは Colab のアイドルタイムアウト（90分）を防ぎます。
# 万が一 cloudflared / Streamlit が落ちた場合は自動再起動し、新しい URL を表示します。

import subprocess, time, re, sys, os
from pathlib import Path
from IPython.display import display, Javascript

PORT = 8501
CLOUDFLARED = Path('/usr/local/bin/cloudflared')

# Colab アイドルタイムアウト防止（55秒ごとに接続ボタンをクリック）
display(Javascript("""
  if (typeof keepAliveInterval === 'undefined') {
    window.keepAliveInterval = setInterval(function() {
      var btn = document.querySelector("#connect");
      if (btn) btn.click();
    }, 55000);
  }
"""))

def is_alive(pid_file):
    try:
        pid = int(Path(pid_file).read_text())
        os.kill(pid, 0)
        return True
    except Exception:
        return False

def restart_streamlit():
    with open('/tmp/streamlit.log', 'a') as log:
        proc = subprocess.Popen(
            [sys.executable, '-m', 'streamlit', 'run', 'app.py',
             '--server.port', str(PORT), '--server.headless', 'true',
             '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
            stdout=log, stderr=log
        )
    Path('/tmp/st_pid').write_text(str(proc.pid))

def restart_cloudflared():
    # stderr はファイルへ（PIPE 不使用）
    Path('/tmp/cloudflared.log').write_text('')
    with open('/tmp/cloudflared.log', 'w') as cf_log:
        proc = subprocess.Popen(
            [str(CLOUDFLARED), 'tunnel', '--url', f'http://localhost:{PORT}'],
            stdout=subprocess.DEVNULL, stderr=cf_log
        )
    Path('/tmp/cf_pid').write_text(str(proc.pid))
    # URL をポーリング
    for _ in range(30):
        time.sleep(1)
        try:
            text = Path('/tmp/cloudflared.log').read_text()
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', text)
            if m:
                url = m.group(0)
                Path('/tmp/app_url').write_text(url)
                return url
        except Exception:
            pass
    return None

current_url = Path('/tmp/app_url').read_text() if Path('/tmp/app_url').exists() else '不明'
print(f'監視開始 | URL: {current_url}')
print('このセルを実行したままにしてください。停止するには ■ を押します。\n')

try:
    tick = 0
    while True:
        time.sleep(30)
        tick += 1

        # Streamlit 監視
        if not is_alive('/tmp/st_pid'):
            print(f'[{tick}] ⚠️  Streamlit 停止 → 再起動中...')
            restart_streamlit()
            time.sleep(5)
            print(f'[{tick}] ✅ Streamlit 再起動完了')

        # cloudflared 監視
        if not is_alive('/tmp/cf_pid'):
            print(f'[{tick}] ⚠️  cloudflared 停止 → 再起動中...')
            new_url = restart_cloudflared()
            print(f'[{tick}] ✅ 新しい URL: {new_url or "取得失敗"}')

        # 1時間ごとに URL を再表示
        if tick % 120 == 0:
            url = Path('/tmp/app_url').read_text()
            print(f'[稼働中 {tick*30//3600}h {(tick*30%3600)//60}m] URL: {url}')

except KeyboardInterrupt:
    print('\n監視を停止しました。プロセスはバックグラウンドで動いています。')
    print('完全に停止する場合は: !pkill -f streamlit; !pkill -f cloudflared')


## トラブルシューティング

| 症状 | 対処 |
|------|------|
| `GPU required` エラー | ランタイム → タイプを変更 → T4 GPU |
| URL が表示されない | Cell 6 を再実行 |
| Cell 7 が止まった | Cell 7 を再実行（バックグラウンドのアプリは継続中） |
| アプリに繋がらない | Cell 6 を再実行（新しい URL が発行される） |
| Colab セッション切れ（90分） | Cell 6 → Cell 7 を再実行 |
| ログ確認 | `!tail -50 /tmp/streamlit.log` |